<a href="https://colab.research.google.com/github/TahaShan16/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TahaShan16/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
%pip -q install duckdb huggingface_hub

In [15]:
import os
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN is missing. Add it in Colab Secrets first."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')",
    "dim_content": "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')",
    "fact_daily": "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')",
}

print("Connected to FlyRank warehouse")

Connected to FlyRank warehouse


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For my refresh / content opportunity lane, one row means one content item for one client in one mid-panel month. I will use March 2026 as the working month because it is inside the panel and avoids the final June 2026 sample month. The source table is fact_content_daily_performance, joined only if needed to dim_clients or dim_content for context. My output is a ranking or score for pages/content items that should be reviewed first. I deliberately exclude June 2026 sample data from label logic because the assignment warns that it is the final month and should not be used to develop labels.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions, and days since last update if available from the content table. These are candidate inputs only when measured before or during the decision window and not derived from the future label.

Label or proxy: a temporary proxy such as next-month decline or needs_review, built from future impressions after the feature window. In this notebook I will sketch the proxy, not treat it as final truth.

Context: client_hash_id, content_hash_id, report_date, month, and content metadata used for grouping, joining, filtering, or explaining rows. IDs are context, not model features.

Excluded: raw URLs, raw queries, client names, private data, the final June 2026 sample month for label development, and any column directly derived from the label. I also exclude trend-style columns from the feature set if they are computed from the same outcome I am trying to predict.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

MONTH = "2026-03"

# Query 1: grain check
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {TABLES["fact_daily"]}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

print("Query 1: grain check")
print("If this returns 0 rows, one row is one client-content-day.")
print("Duplicate grain rows found:", len(grain_check))
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1: grain check
If this returns 0 rows, one row is one client-content-day.
Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,row_count


In [19]:
# Query 2: row count and date span for the March 2026 lane slice
slice_summary = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES["fact_daily"]}
    WHERE month = '{MONTH}'
""").df()

print("Query 2: March 2026 slice count and date span")
slice_summary

Query 2: March 2026 slice count and date span


,rows,clients,content_items,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [20]:
# Query 3: availability check using IS TRUE
availability_summary = con.sql(f"""
    SELECT
        COUNT(*) AS rows_in_month,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_gsc,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
        SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_both
    FROM {TABLES["fact_daily"]}
    WHERE month = '{MONTH}'
""").df()

print("Query 3: availability checked with IS TRUE")
availability_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3: availability checked with IS TRUE


,rows_in_month,rows_with_gsc,rows_with_ga4,rows_with_both
0,9841378,3611061.0,413966.0,364347.0


The grain check returned 0 duplicate rows for report_date, client_hash_id, and content_hash_id, so the daily fact table behaves as one row per client-content-day for this month. For March 2026, the slice has 9,841,378 rows, 55 clients, 331,437 content items, and dates from 2026-03-01 to 2026-03-31. Availability is not complete: 3,611,061 rows have GSC data, 413,966 rows have GA4 data, and 364,347 rows have both when checked with IS TRUE. That means the feature frame should filter explicitly for available data instead of treating unavailable rows as real zeros.

In [21]:
# Five March features + April decline label

feature_frame = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_march,
            SUM(gsc_clicks) AS clicks_march,
            AVG(gsc_avg_position) AS avg_position_march,
            SUM(ga4_sessions) AS sessions_march,
            SUM(ga4_engaged_sessions) AS engaged_sessions_march
        FROM {TABLES["fact_daily"]}
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),
    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_april
        FROM {TABLES["fact_daily"]}
        WHERE month = '2026-04'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.impressions_march,
        m.clicks_march,
        100.0 * m.clicks_march / NULLIF(m.impressions_march, 0) AS ctr_march,
        m.avg_position_march,
        m.sessions_march,
        100.0 * m.engaged_sessions_march / NULLIF(m.sessions_march, 0) AS engagement_rate_march,
        a.impressions_april,
        CASE
            WHEN a.impressions_april < 0.8 * m.impressions_march THEN 1
            ELSE 0
        END AS proxy_needs_review
    FROM march m
    INNER JOIN april a
      ON m.client_hash_id = a.client_hash_id
     AND m.content_hash_id = a.content_hash_id
""").df()

feature_frame["leaky_label_copy"] = feature_frame["proxy_needs_review"]

print("Feature frame rows:", len(feature_frame))
print("Proxy positive rows:", int(feature_frame["proxy_needs_review"].sum()))
print("Proxy positive rate:", round(feature_frame["proxy_needs_review"].mean(), 3))

feature_frame.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame rows: 32567
Proxy positive rows: 1721
Proxy positive rate: 0.053


,client_hash_id,content_hash_id,impressions_march,clicks_march,ctr_march,avg_position_march,sessions_march,engagement_rate_march,impressions_april,proxy_needs_review,leaky_label_copy
0,client_9958f0a7ae1df715,content_3c3b575d53a71932,1636.0,1.0,0.061125,9.702400,24.0,0.000000,373.0,1,1
1,client_9958f0a7ae1df715,content_d8288f22519f7d53,1193.0,0.0,0.000000,22.574773,120.0,3.333333,464.0,1,1
2,client_9958f0a7ae1df715,content_d18b0265046ac49a,303.0,3.0,0.990099,6.198434,14.0,7.142857,113.0,1,1
3,client_9958f0a7ae1df715,content_1824be59b01d64cf,197.0,2.0,1.015228,27.506212,11.0,9.090909,520.0,0,0
4,client_9958f0a7ae1df715,content_4e2075ce500c657b,129.0,2.0,1.550388,6.880029,9.0,0.000000,89.0,1,1
5,client_9958f0a7ae1df715,content_c040204537f06a59,1562.0,8.0,0.512164,10.501741,31.0,3.225806,1423.0,0,0
6,client_9958f0a7ae1df715,content_986d48a790f0f5ad,806.0,3.0,0.372208,12.978865,29.0,3.448276,139.0,1,1
7,client_9958f0a7ae1df715,content_243bf9f16639bb7d,128.0,4.0,3.125000,21.191773,6.0,16.666667,213.0,0,0
8,client_9958f0a7ae1df715,content_43e2f39f8ecdf102,241.0,0.0,0.000000,8.086403,4.0,0.000000,342.0,0,0
9,client_9958f0a7ae1df715,content_a0144defc87d8b23,301.0,0.0,0.000000,48.664262,13.0,0.000000,456.0,0,0


Five features and why they are knowable at the decision moment:

1. impressions_march: knowable at the decision moment because it is observed search visibility from the March 2026 working month.
2. clicks_march: knowable at the decision moment because it is observed Search Console click volume from the same closed month.
3. ctr_march: knowable at the decision moment because it is calculated from March clicks divided by March impressions.
4. avg_position_march: knowable at the decision moment because it summarizes observed March ranking position.
5. engagement_rate_march: knowable at the decision moment because it is calculated from March GA4 engaged sessions divided by March sessions, after filtering for ga4_data_available IS TRUE.

The proxy target is proxy_needs_review. It is not an honest future label yet because this notebook is focused on the data contract and feature leakage lesson. The leaky_label_copy column is included only on purpose for the trap experiment and must not be kept as a real feature.

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score

honest_features = [
    "impressions_march",
    "clicks_march",
    "ctr_march",
    "avg_position_march",
    "engagement_rate_march",
]

leaky_features = honest_features + ["leaky_label_copy"]

model_df = feature_frame.dropna(subset=honest_features + ["proxy_needs_review"]).copy()

X_honest = model_df[honest_features]
X_leaky = model_df[leaky_features]
y = model_df["proxy_needs_review"]

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_honest, y, test_size=0.25, random_state=42, stratify=y
)

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky, y, test_size=0.25, random_state=42, stratify=y
)

honest_model = DecisionTreeClassifier(max_depth=3, random_state=42)
leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42)

honest_model.fit(Xh_train, yh_train)
leaky_model.fit(Xl_train, yl_train)

honest_pred = honest_model.predict(Xh_test)
leaky_pred = leaky_model.predict(Xl_test)

print("Honest model")
print("Accuracy:", round(accuracy_score(yh_test, honest_pred), 3))
print("Precision:", round(precision_score(yh_test, honest_pred), 3))

print("\nLeaky model with label-derived column")
print("Accuracy:", round(accuracy_score(yl_test, leaky_pred), 3))
print("Precision:", round(precision_score(yl_test, leaky_pred), 3))

feature_frame = feature_frame.drop(columns=["leaky_label_copy"])

print("\nRemoved leaky_label_copy. Columns kept:")
print(feature_frame.columns.tolist())

Honest model
Accuracy: 0.947
Precision: 0.0

Leaky model with label-derived column
Accuracy: 1.0
Precision: 1.0

Removed leaky_label_copy. Columns kept:
['client_hash_id', 'content_hash_id', 'impressions_march', 'clicks_march', 'ctr_march', 'avg_position_march', 'sessions_march', 'engagement_rate_march', 'impressions_april', 'proxy_needs_review']


The leakage trap is visible here. When I include leaky_label_copy, the model gets perfect accuracy and precision because that column is just a copy of the label. After removing it, the honest model has accuracy of 0.947 but precision of 0.0. This means the high accuracy is misleading: because only about 5.3% of rows are positive, a model can look accurate while failing to identify the pages that actually need review. For this lane, precision near the top of a ranked queue would matter more than overall accuracy.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One limitation of this slice is that it only uses March 2026 features and an April 2026 proxy outcome. This is useful for checking the data contract, but it does not prove that a refresh would cause improvement. Another limitation is that availability is uneven: many March rows do not have both GSC and GA4 data, so filtering for both may bias the feature frame toward clients or content items with better tracking coverage.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.